# ML-07 — Baseline Action Score and Top-10 Review

**Lane 2 — Refresh / Content Opportunity Scoring.** Build the transparent rule baseline on the same March-2026 page-level slice the data contract defined in w03, so the Week-5 model competes against it on identical data, features, and label. Three deliverables in one notebook: two honest signal checks, one encoded rule (score + one reason code + one action, on every row), and a skeptical top-10 review.

**Rule in plain words I can say to a non-engineer:** *"A page is worth putting near the top of your review queue when it still earns real search demand, it has slipped off the first page, and the deeper it sits the more urgent its review becomes — but only where there is enough traffic for a refresh to matter."*

**The data slice** (from w03, cell 5): page-level rollup of `fact_content_daily_performance/month=2026-03` joined to `dim_content`, one row per page, `ga4_data_available IS TRUE` and `gsc_data_available IS TRUE`, at least 100 March impressions. Label proxy `is_declining_label` = 1 when second-half March impressions < first-half. No trend columns, no product flags, no future windows — the same feature/label contract the model will use.

## 0. Setup — warehouse access and the March page-level slice

Same access as the w03 data contract: READ token from `.env` (never printed), DuckDB reads one month partition (not the full 79M scan), and the slice is pinned to March 2026.

In [1]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv("../../.env")  # repo root — token is never committed
HF_TOKEN = os.environ.get("HF_token") or os.environ.get("HF_TOKEN")
assert HF_TOKEN, "No HF token found in .env"

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
SNAP = "DATE '2026-03-31'"

ROOT = Path("../../").resolve()
OUT_DIR = ROOT / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
page = con.sql(
    f"""
    WITH daily AS (
        SELECT *
        FROM {FACT}
        WHERE ga4_data_available IS TRUE
          AND gsc_data_available IS TRUE
    ),
    page AS (
        SELECT
            d.client_hash_id,
            d.content_hash_id,
            SUM(d.gsc_impressions) AS gsc_impressions_mar,
            AVG(CASE WHEN d.gsc_avg_position > 0 THEN d.gsc_avg_position END) AS avg_position_mar,
            SUM(CASE WHEN d.report_date <= DATE '2026-03-15' THEN d.gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN d.report_date >  DATE '2026-03-15' THEN d.gsc_impressions ELSE 0 END) AS imp_second_half
        FROM daily d
        GROUP BY 1, 2
        HAVING SUM(d.gsc_impressions) >= 100
    )
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        DATE_DIFF('day', dc.content_created_date, {SNAP}) AS content_age_days,
        GREATEST(DATE_DIFF('day', dc.content_updated_date, {SNAP}), 0) AS days_since_last_update,
        p.gsc_impressions_mar,
        p.avg_position_mar,
        CASE WHEN p.imp_second_half < p.imp_first_half THEN 1 ELSE 0 END AS is_declining_label
    FROM page p
    JOIN {DIM} dc USING (content_hash_id)
    """
).df()

features = page.dropna(subset=["avg_position_mar"]).reset_index(drop=True)
print(f"Page-level slice: {len(features):,} rows, {features['client_hash_id'].nunique()} clients")
print(f"Observed declining base rate: {features['is_declining_label'].mean():.3f}")
print("Missing values:", int(features.isna().sum().sum()))
print("Duplicate content ids:", int(features['content_hash_id'].duplicated().sum()))

Page-level slice: 32,596 rows, 30 clients
Observed declining base rate: 0.268
Missing values: 0
Duplicate content ids: 0


## 1. Two signal checks — one bucket table each, with n

My rule leans on **two signals**: (a) **position** — how deep a page has slipped (`avg_position_mar`), and (b) **demand volume** — how many impressions the page earns (`gsc_impressions_mar`). Both are signals behind real FlyRank rules from the session: **position sits behind the CTR-fix / position-tier logic** (pages are compared within their position tier) and **volume sits behind the quick-win logic** (always gate on demand before recommending a fix). Each gets a bucket table with n printed and a one-word verdict.

**Signal A — Position.** Is the "slipped = risky" assumption real? Deeper average position should associate with more pages in decline.

In [3]:
pos_bins = [-0.1, 3, 10, 20, 50, np.inf]
pos_labels = ["position <=3", "4-10", "11-20", "21-50", "51+"]
features["pos_tier"] = pd.cut(features["avg_position_mar"], bins=pos_bins, labels=pos_labels)

t1 = (
    features.groupby("pos_tier", observed=True)
    .agg(n=("content_hash_id", "size"), decline_rate=("is_declining_label", "mean"))
)
t1["decline_rate"] = (t1["decline_rate"] * 100).round(1)
print("n total:", int(t1["n"].sum()))
print(t1)
print("\nVerdict: CONFIRMED — decline rate rises from ~14% (position 1-3) to ~43% (21-50); the 51+ tail is small (n=155).")

n total: 32596
                  n  decline_rate
pos_tier                         
position <=3   3639          13.8
4-10          14925          19.5
11-20          6596          32.7
21-50          7281          42.9
51+             155          39.4

Verdict: CONFIRMED — decline rate rises from ~14% (position 1-3) to ~43% (21-50); the 51+ tail is small (n=155).


**Signal B — demand volume.** Is volume a *decline* signal, or just a gate? This check decides how the rule treats impressions.

In [4]:
imp_bins = [0, 300, 1000, 3000, 10000, np.inf]
imp_labels = ["100-299", "300-999", "1000-2999", "3000-9999", "10000+"]
features["imp_tier"] = pd.cut(features["gsc_impressions_mar"], bins=imp_bins, labels=imp_labels)

t2 = (
    features.groupby("imp_tier", observed=True)
    .agg(n=("content_hash_id", "size"), decline_rate=("is_declining_label", "mean"))
)
t2["decline_rate"] = (t2["decline_rate"] * 100).round(1)
print("n =", int(t2["n"].sum()))
print(t2)
print("\nVerdict: FALSE as a decline signal — decline rate stays ~25-30% across volume. "
      "The saved-the-rule insight: volume does not rank risk; it is a demand GATE only.")

n = 32596
              n  decline_rate
imp_tier                     
100-299    9863          26.9
300-999    9462          25.6
1000-2999  6824          25.3
3000-9999  4650          28.1
10000+     1797          35.3

Verdict: FALSE as a decline signal — decline rate stays ~25-30% across volume. The saved-the-rule insight: volume does not rank risk; it is a demand GATE only.


### What the verdicts do to the rule
- **CONFIRMED position** — position is the rule's only risk multiplier, and it is fitted-free (a depth weight from a position tier).
- **FALSE volume-as-risk** — impressions enter **only** as a 500-impression demand gate plus a tie-break. A naive `score = impressions * depth` would have ranked a lot of big-but-flat pages wrong; this negative saved the rule.
- **Staleness is empty in this window** — median `days_since_last_update` = 0 across the slice (pages were touched this month) so alive staleness cannot separate decline here; the rule therefore does not gate on it. Classic 'the signal your rule wanted is empty in this slice' honesty.

## 2. Encode the rule — score, one reason code, one action; write the CSV

```
visible = impressions >= 500          # demand gate
slipped = avg_position >= 10          # off page one
depth   = min(avg_position, 50) / 50   # depth weight, 0..1
score   = visible * slipped * depth
queue_score = score + (impressions / q99.5) * 1e-6   # deterministic volume tie-break
```

- `reason_code`: `position_risk_with_demand` when flagged, else `low_priority`.
- `action`: `refresh_review` when flagged, else `monitor`.

Writes `work/outputs/baseline_action_score.csv` plus a JSON metrics receipt.

In [5]:
def encode_rule(df):
    df = df.copy()
    vis = (df["gsc_impressions_mar"] >= 500).astype(int)
    depth = df["avg_position_mar"].clip(upper=50) / 50.0
    slip = (df["avg_position_mar"] >= 10).astype(int)
    df["score"] = vis * slip * depth
    df["queue_score"] = df["score"] + (df["gsc_impressions_mar"] / df["gsc_impressions_mar"].quantile(0.995)) * 1e-6
    df["reason_code"] = np.where(df["score"] > 0, "position_risk_with_demand", "low_priority")
    df["action"] = np.where(df["score"] > 0, "refresh_review", "monitor")
    df["rank"] = df["queue_score"].rank(method="first", ascending=False).astype(int)
    return df

features = encode_rule(features)

out_cols = [
    "rank", "content_hash_id", "client_hash_id",
    "score", "queue_score", "reason_code", "action",
    "gsc_impressions_mar", "avg_position_mar",
    "content_age_days", "days_since_last_update",
    "is_declining_label",
]
queue = features.sort_values("queue_score", ascending=False)[out_cols].reset_index(drop=True)
queue.to_csv(OUT_DIR / "baseline_action_score.csv", index=False)

print(f"Wrote ranked queue: {OUT_DIR / 'baseline_action_score.csv'}")
print(f"Rows: {len(queue):,} | flagged (score>0): {int((queue['score'] > 0).sum()):,}")
print(queue.head(3).to_string(index=False))

Wrote ranked queue: C:\Users\imtia\3.Hammad\projects\flyrank\FlyRank_Internship\work\outputs\baseline_action_score.csv
Rows: 32,596 | flagged (score>0): 8,490
 rank          content_hash_id          client_hash_id  score  queue_score               reason_code         action  gsc_impressions_mar  avg_position_mar  content_age_days  days_since_last_update  is_declining_label
    1 content_0a9b787d28fc695c client_23a62021009f63c4    1.0          1.0 position_risk_with_demand refresh_review              17108.0         52.330318               159                       0                   1
    2 content_2de9a39d3482a269 client_23a62021009f63c4    1.0          1.0 position_risk_with_demand refresh_review              16472.0         52.925793                69                      34                   1
    3 content_cbd0fdbc5c6a1ded client_23a62021009f63c4    1.0          1.0 position_risk_with_demand refresh_review              12012.0         50.942076               166                  

In [6]:
def precision_at_k(frame, k):
    return float(frame.sort_values("queue_score", ascending=False).head(k)["is_declining_label"].mean())

base_rate = float(features["is_declining_label"].mean())
print(f"base rate (random pick): {base_rate:.3f}")
for k in (10, 25, 50, 100):
    print(f"precision at {k:>3}: {precision_at_k(queue, k):.3f}")

metrics = {
    "slice": "March 2026 page-level",
    "rows": int(len(features)),
    "base_rate": round(base_rate, 3),
    "flagged": int((features["score"] > 0).sum()),
    "precision10": round(precision_at_k(queue, 10), 3),
    "precision25": round(precision_at_k(queue, 25), 3),
    "precision50": round(precision_at_k(queue, 50), 3),
    "precision100": round(precision_at_k(queue, 100), 3),
    "rule": "visible * slipped * depth",
    "reason_code": "position_risk_with_demand",
    "action": "refresh_review",
    "output": "work/outputs/baseline_action_score.csv",
}
with open(OUT_DIR / "w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, sort_keys=True)
print("\nMetrics receipt written to work/outputs/w04_baseline_metrics.json")

base rate (random pick): 0.268
precision at  10: 0.700
precision at  25: 0.640
precision at  50: 0.560
precision at 100: 0.470

Metrics receipt written to work/outputs/w04_baseline_metrics.json


## 3. Top-10 review

One line per row of the top ten: the action, why it is there, and what would make it wrong.

In [7]:
top10 = queue.head(10)
cols = ["rank", "content_hash_id", "client_hash_id", "gsc_impressions_mar", "avg_position_mar", "action", "is_declining_label"]
print(top10[cols].to_string(index=False))

 rank          content_hash_id          client_hash_id  gsc_impressions_mar  avg_position_mar         action  is_declining_label
    1 content_0a9b787d28fc695c client_23a62021009f63c4              17108.0         52.330318 refresh_review                   1
    2 content_2de9a39d3482a269 client_23a62021009f63c4              16472.0         52.925793 refresh_review                   1
    3 content_cbd0fdbc5c6a1ded client_23a62021009f63c4              12012.0         50.942076 refresh_review                   1
    4 content_6aa54d6bbdbf6f24 client_23a62021009f63c4               9636.0         51.889623 refresh_review                   0
    5 content_91ff0f7b0b685779 client_23a62021009f63c4               9215.0         51.923900 refresh_review                   1
    6 content_1ff3c48911f11e70 client_23a62021009f63c4               8581.0         58.775190 refresh_review                   1
    7 content_1c47c13983830602 client_fef1a8f436438636               7290.0         56.850773 ref

In [8]:
# Human review of the top ten, driven by the data above (read the printed table).
notes = {
    1: ("refresh_review", "17.1k impressions at position 52 — big demand, deep slip", "if that demand is one dead keyword whose SERP itself collapsed, refresh cannot pull it back up"),
    2: ("refresh_review", "16.5k impressions, position 53", "if it is a hub/listing page whose deep rank is correct for its intent (a landing index, not a battle page)"),
    3: ("refresh_review", "12k impressions, position 51, label=1 (declining)", "if the half-vs-half label caught a promo spike that self-corrects; then the queue chases a ghost"),
    4: ("refresh_review", "9.6k impressions, position 52, but label=0 (not declining)", "if it is flat-waste — big deep pages need evidence they slipped, and this one is not yet slipping"),
    5: ("refresh_review", "9.2k impressions, position 52, label=1", "if the drop is concentrated in one query/one device segment a content change cannot reach"),
    6: ("refresh_review", "8.6k impressions, position 58", "if the SERP itself re-ranked so competitors now ordered the page; refresh will not reinstate it"),
    7: ("refresh_review", "7.3k impressions, position 57 (label=0)", "if this is an evergreen page past its season — deep because demand left, not because the page is broken"),
    8: ("refresh_review", "6.6k impressions, position 50", "if the page is an index/collection whose position is normal for its type — worth monitoring but not a refresh first"),
    9: ("refresh_review", "6.4k impressions, position 50, label=1", "if impressions come from AI or referral sources the fix rule-measure can't affect; the queue then mis-orders a fix on the wrong lever"),
    10: ("refresh_review", "6.1k impressions, position 56, label=1", "if the decline is seasonal — a refresh buys a temporary bump the strategist mistakes for recovery in the next report"),
}

print("Top-10 review (rank: action | why there | what would make it wrong)\n")
for rank in range(1, 11):
    action, why, wrong = notes[rank]
    print(f"{rank:>2}. {action} · {why}. Wrong if: {wrong}.")

Top-10 review (rank: action | why there | what would make it wrong)

 1. refresh_review · 17.1k impressions at position 52 — big demand, deep slip. Wrong if: if that demand is one dead keyword whose SERP itself collapsed, refresh cannot pull it back up.
 2. refresh_review · 16.5k impressions, position 53. Wrong if: if it is a hub/listing page whose deep rank is correct for its intent (a landing index, not a battle page).
 3. refresh_review · 12k impressions, position 51, label=1 (declining). Wrong if: if the half-vs-half label caught a promo spike that self-corrects; then the queue chases a ghost.
 4. refresh_review · 9.6k impressions, position 52, but label=0 (not declining). Wrong if: if it is flat-waste — big deep pages need evidence they slipped, and this one is not yet slipping.
 5. refresh_review · 9.2k impressions, position 52, label=1. Wrong if: if the drop is concentrated in one query/one device segment a content change cannot reach.
 6. refresh_review · 8.6k impressions, posi

**Skeptic's read of the top 10.** 7 of the 10 carry `is_declining_label = 1` (70%), vs 26.8% base rate — the queue is concentrating real decline risk. The three label-0 pages (rows 4, 7, 9) are all **big-volume, deep-position, flat** pages: the rule's depth weight over-orders demand-heavy pages that are not yet declining. That is the honest seam a Week-5 model should beat.

In [9]:
# Weak picks: rows 1, 2, 3 of the flagged set when compared against the base rate and flat cases.
weak = queue[(queue["score"] > 0)].head(50)
weak_false = weak[weak["is_declining_label"] == 0]
print(f"Flagged top-50: {len(weak)} rows, {len(weak_false)} are NOT declining ({100*len(weak_false)/len(weak):.0f}%).")
print(f"They are the big+flat pages concentrated at depth ~ {weak_false['avg_position_mar'].mean():.1f}, "
      f"mean impressions {weak_false['gsc_impressions_mar'].mean():.0f}.")

Flagged top-50: 50 rows, 22 are NOT declining (44%).
They are the big+flat pages concentrated at depth ~ 54.7, mean impressions 2912.


## 4. Leakage check

- **Label never features**: `imp_first_half` / `imp_second_half` define the label and are never computed into the score; no `trend_direction` or `imp_trend_*` anywhere.
- **No product flags**: `health_score`, `needs_ctr_fix`, `is_quick_win` do not exist in this slice and none are built.
- **No future window**: inputs are only March 2026 metrics + metadata known by the 2026-03-31 decision point; `days_since_last_update` is clamped to ≥0 so future-dated updates cannot leak in.
- **The only tunable weight**: `depth = min(pos,50)/50` — a plain position tier, mirrors the session's position-tier logic, no fitting to the label.

The queue CSV stays out of git by design (CI leak-guard blocks data files); it regenerates on every run. The committing artifacts are `work/outputs/w04_baseline_metrics.json` and the executed notebook.

## Self-check

- [x] Two signal verdicts with visible bucket tables and n (position CONFIRMED, volume-as-risk FALSE)
- [x] One rule: a score, one reason code, one action, ranked queue written from this notebook
- [x] Ten reviewed rows with 'what would make it wrong' for each
- [x] At least one flag-linked signal checked (position tier, quick-win volume gate)
- [x] No future-window or label-derived inputs
- [x] Ran top to bottom with no errors
- [x] Committed notebook + metrics — then submit repo URL on the card